## RQ3 Data Preparation: Multi-Year HMDA Dataset

This section prepares the dataset for Research Question 3.

RQ3 asks whether the determinants of mortgage approval and the magnitude of demographic disparities are stable over time. To answer this question, we need a multi-year dataset rather than a single-year cross-section.

We therefore combine HMDA raw data from 2022 to 2025, clean the key variables, and construct a final dataset suitable for regression analysis.

In [30]:
import os
import pandas as pd
import numpy as np

### Step 1 | Set File Paths

The raw HMDA files are stored in the Downloads folder. We first define the file paths and confirm that the files exist.

In [31]:
downloads_path = os.path.expanduser("~/Downloads")

files = {
    2022: os.path.join(downloads_path, "2022.csv"),
    2023: os.path.join(downloads_path, "2023.csv"),
    2024: os.path.join(downloads_path, "2024.csv"),
    2025: os.path.join(downloads_path, "2025.csv"),
}

print("Downloads path:", downloads_path)
print("\nFiles to load:")
for yr, path in files.items():
    print(f"{yr} -> {path} | Exists: {os.path.exists(path)}")

Downloads path: /Users/brian/Downloads

Files to load:
2022 -> /Users/brian/Downloads/2022.csv | Exists: True
2023 -> /Users/brian/Downloads/2023.csv | Exists: True
2024 -> /Users/brian/Downloads/2024.csv | Exists: True
2025 -> /Users/brian/Downloads/2025.csv | Exists: True


### Step 2 | Define a Helper Function for Column Names

Because raw HMDA files may contain formatting inconsistencies in column names, we standardize all column names by converting them to lowercase and replacing spaces or special characters with underscores.

In [32]:
def clean_column_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
    )
    return df

### Step 3 | Preview the Raw Data Structure

The HMDA raw files are pipe-delimited rather than comma-delimited. We preview the first file to inspect the available columns.

In [33]:
preview_df = pd.read_csv(files[2022], sep="|", nrows=5, low_memory=False)
preview_df = clean_column_names(preview_df)

print("Columns in 2022 file:")
print(preview_df.columns.tolist())

Columns in 2022 file:
['activity_year', 'lei', 'loan_type', 'loan_purpose', 'preapproval', 'construction_method', 'occupancy_type', 'loan_amount', 'action_taken', 'state_code', 'county_code', 'census_tract', 'applicant_ethnicity_1', 'applicant_ethnicity_2', 'applicant_ethnicity_3', 'applicant_ethnicity_4', 'applicant_ethnicity_5', 'co_applicant_ethnicity_1', 'co_applicant_ethnicity_2', 'co_applicant_ethnicity_3', 'co_applicant_ethnicity_4', 'co_applicant_ethnicity_5', 'applicant_ethnicity_observed', 'co_applicant_ethnicity_observed', 'applicant_race_1', 'applicant_race_2', 'applicant_race_3', 'applicant_race_4', 'applicant_race_5', 'co_applicant_race_1', 'co_applicant_race_2', 'co_applicant_race_3', 'co_applicant_race_4', 'co_applicant_race_5', 'applicant_race_observed', 'co_applicant_race_observed', 'applicant_sex', 'co_applicant_sex', 'applicant_sex_observed', 'co_applicant_sex_observed', 'applicant_age', 'applicant_age_above_62', 'co_applicant_age', 'co_applicant_age_above_62', 'inc

### Step 4 | Select Relevant Variables

To keep the analysis focused and manageable, we retain only the variables that are relevant for the RQ3 approval model and are broadly consistent with the earlier research question structure.

In [34]:
target_cols = [
    "activity_year",
    "loan_type",
    "loan_purpose",
    "preapproval",
    "construction_method",
    "occupancy_type",
    "loan_amount",
    "action_taken",
    "property_value",
    "loan_term",
    "interest_rate",
    "rate_spread",
    "hoepa_status",
    "lien_status",
    "income",
    "applicant_race_1",
    "applicant_sex",
    "derived_age",
    "conforming_loan_limit",
    "debt_to_income_ratio",
    "applicant_credit_score_type",
    "aus_1"
]

print("Number of target columns:", len(target_cols))
print(target_cols)

Number of target columns: 22
['activity_year', 'loan_type', 'loan_purpose', 'preapproval', 'construction_method', 'occupancy_type', 'loan_amount', 'action_taken', 'property_value', 'loan_term', 'interest_rate', 'rate_spread', 'hoepa_status', 'lien_status', 'income', 'applicant_race_1', 'applicant_sex', 'derived_age', 'conforming_loan_limit', 'debt_to_income_ratio', 'applicant_credit_score_type', 'aus_1']


### Step 5 | Load and Combine the Four Years of HMDA Data

Because the HMDA raw files are very large, we read them in chunks rather than loading each full file into memory at once. To make the notebook faster and more manageable, we also take a 10% random sample from each chunk.

This approach preserves the multi-year structure while making the computation feasible in Jupyter.

In [35]:
dfs = []

for year, path in files.items():
    print(f"\nLoading {year}...")
    
    chunk_list = []
    
    for chunk in pd.read_csv(
        path,
        sep="|",
        chunksize=50000,
        low_memory=False
    ):
        chunk = clean_column_names(chunk)
        
        # keep only columns that exist in this chunk
        keep_cols = [col for col in target_cols if col in chunk.columns]
        chunk = chunk[keep_cols].copy()
        
        if "activity_year" not in chunk.columns:
            chunk["activity_year"] = year
        
        # sample 10% for speed
        chunk = chunk.sample(frac=0.10, random_state=42)
        
        chunk_list.append(chunk)
    
    year_df = pd.concat(chunk_list, ignore_index=True)
    dfs.append(year_df)
    
    print(f"{year} shape:", year_df.shape)
    print("Race column exists:", "applicant_race_1" in year_df.columns)
    print("Sex column exists:", "applicant_sex" in year_df.columns)

df = pd.concat(dfs, ignore_index=True)

print("\nCombined shape:", df.shape)
print(df.columns.tolist())


Loading 2022...
2022 shape: (1612598, 19)
Race column exists: True
Sex column exists: True

Loading 2023...
2023 shape: (1157523, 19)
Race column exists: True
Sex column exists: True

Loading 2024...
2024 shape: (1225913, 19)
Race column exists: True
Sex column exists: True

Loading 2025...
2025 shape: (1353753, 19)
Race column exists: True
Sex column exists: True

Combined shape: (5349787, 19)
['activity_year', 'loan_type', 'loan_purpose', 'preapproval', 'construction_method', 'occupancy_type', 'loan_amount', 'action_taken', 'property_value', 'loan_term', 'interest_rate', 'rate_spread', 'hoepa_status', 'lien_status', 'income', 'applicant_race_1', 'applicant_sex', 'debt_to_income_ratio', 'aus_1']


**Observations:** The four yearly files have now been combined into a single multi-year dataset. This is necessary because RQ3 focuses on whether approval patterns and demographic disparities are stable over time.

In [36]:
print(df.columns.tolist())
print("applicant_race_1" in df.columns)
print("applicant_sex" in df.columns)

['activity_year', 'loan_type', 'loan_purpose', 'preapproval', 'construction_method', 'occupancy_type', 'loan_amount', 'action_taken', 'property_value', 'loan_term', 'interest_rate', 'rate_spread', 'hoepa_status', 'lien_status', 'income', 'applicant_race_1', 'applicant_sex', 'debt_to_income_ratio', 'aus_1']
True
True


In [37]:
if "derived_race" not in df.columns:
    print("Creating derived_race from applicant_race_1...")
    if "applicant_race_1" in df.columns:
        df["derived_race"] = df["applicant_race_1"].astype(str)
    else:
        print("WARNING: applicant_race_1 not found")

if "derived_sex" not in df.columns:
    print("Creating derived_sex from applicant_sex...")
    if "applicant_sex" in df.columns:
        df["derived_sex"] = df["applicant_sex"].astype(str)
    else:
        print("WARNING: applicant_sex not found")

print("derived_race exists:", "derived_race" in df.columns)
print("derived_sex exists:", "derived_sex" in df.columns)

Creating derived_race from applicant_race_1...
Creating derived_sex from applicant_sex...
derived_race exists: True
derived_sex exists: True


### Step 6 | Construct Demographic Variables

The raw HMDA files do not always include the cleaned `derived_race` and `derived_sex` variables. Therefore, when these variables are unavailable, we construct demographic variables using the primary applicant demographic fields available in the raw data.

This allows us to preserve the demographic dimension of the approval analysis while keeping the model specification as consistent as possible across years.

In [38]:
required_cols = [
    "activity_year",
    "action_taken",
    "income",
    "loan_amount",
    "derived_race",
    "derived_sex"
]

missing_required = [c for c in required_cols if c not in df.columns]

print("Missing required columns:", missing_required)

if len(missing_required) > 0:
    raise ValueError(f"Missing required columns: {missing_required}")

Missing required columns: []


### Step 7 | Construct the Approval Outcome Variable

We define a binary approval variable based on HMDA `action_taken`.

For this analysis:
- `1` = loan originated
- `2` = application approved but not accepted
- `3` = application denied

We keep only these three outcomes and code approval as 1 for values 1 and 2, and 0 for value 3.

In [39]:
df["action_taken"] = pd.to_numeric(df["action_taken"], errors="coerce")

df = df[df["action_taken"].isin([1, 2, 3])].copy()
df["approved"] = np.where(df["action_taken"].isin([1, 2]), 1, 0)

print("Action taken counts:")
print(df["action_taken"].value_counts().sort_index())

print("\nApproved counts:")
print(df["approved"].value_counts())

print("\nApproval rate:")
print(round(df["approved"].mean(), 4))

Action taken counts:
action_taken
1    2716561
2     151087
3     877717
Name: count, dtype: int64

Approved counts:
approved
1    2867648
0     877717
Name: count, dtype: int64

Approval rate:
0.7657


**Observations:** The binary dependent variable `approved` captures whether an application was approved or denied. This variable will serve as the outcome in the logistic regression models used later in RQ3.

### Step 8 | Clean Numeric Variables

Some HMDA fields contain strings such as `Exempt` or `NA`. We convert these entries into missing values and then convert the selected numeric variables into numeric format.

In [40]:
na_values = ["NA", "Exempt", "8888", "9999", "", "nan", "NaN", "None"]

num_cols = [
    "income",
    "loan_amount",
    "property_value",
    "interest_rate",
    "rate_spread",
    "loan_term"
]

for col in num_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).replace(na_values, np.nan)
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data types after cleaning:")
for col in num_cols:
    if col in df.columns:
        print(col, "->", df[col].dtype)

Data types after cleaning:
income -> float64
loan_amount -> int64
property_value -> float64
interest_rate -> float64
rate_spread -> float64
loan_term -> float64


### Step 9 | Create Ordinal Variables

To remain consistent with the cleaned dataset style used previously, we create simple ordinal string versions of debt-to-income ratio and age.

In [41]:
# only add age if it exists
if "age_ordinal" in df.columns:
    model_cols.append("age_ordinal")


available_model_cols = [
    col for col in model_cols
    if col in df.columns
]

rq3_df = df[available_model_cols].dropna().copy()

print("Final shape:", rq3_df.shape)
print(rq3_df.columns.tolist())

Final shape: (3359575, 14)
['activity_year', 'approved', 'income', 'loan_amount', 'preapproval', 'derived_race', 'derived_sex', 'loan_type', 'loan_purpose', 'lien_status', 'construction_method', 'occupancy_type', 'property_value', 'aus_1']


### Step 10 | Keep the Final Model Variables

We now retain the variables intended for the RQ3 model and remove rows with missing values in those fields.

In [42]:
model_cols = [
    "activity_year",
    "approved",
    "income",
    "loan_amount",
    "preapproval",
    "derived_race",
    "derived_sex",
    "loan_type",
    "loan_purpose",
    "lien_status",
    "construction_method",
    "occupancy_type",
    "property_value",
    "conforming_loan_limit",
    "aus_1",
    "dti_ordinal",
    "age_ordinal"
]

available_model_cols = [col for col in model_cols if col in df.columns]
missing_model_cols = [col for col in model_cols if col not in df.columns]

print("Available model columns:")
print(available_model_cols)

print("\nMissing model columns:")
print(missing_model_cols)

rq3_df = df[available_model_cols].dropna().copy()

print("\nFinal cleaned sample shape:")
print(rq3_df.shape)

print("\nYear counts in final sample:")
print(rq3_df["activity_year"].value_counts().sort_index())

print("\nFinal approval rate:")
print(round(rq3_df["approved"].mean(), 4))

print("\nPreview:")
print(rq3_df.head())

Available model columns:
['activity_year', 'approved', 'income', 'loan_amount', 'preapproval', 'derived_race', 'derived_sex', 'loan_type', 'loan_purpose', 'lien_status', 'construction_method', 'occupancy_type', 'property_value', 'aus_1']

Missing model columns:
['conforming_loan_limit', 'dti_ordinal', 'age_ordinal']

Final cleaned sample shape:
(3359575, 14)

Year counts in final sample:
activity_year
2022    1037974
2023     731925
2024     771126
2025     818550
Name: count, dtype: int64

Final approval rate:
0.7736

Preview:
   activity_year  approved  income  loan_amount  preapproval derived_race  \
1           2022         1    77.0       115000            2          5.0   
2           2022         1    83.0       105000            2          5.0   
3           2022         1    32.0        85000            2          3.0   
4           2022         1   126.0       225000            2          5.0   
5           2022         1    54.0        25000            2          5.0   

  d

**Observations:** The cleaned multi-year dataset is now ready for RQ3. It includes the binary approval variable, borrower characteristics, loan characteristics, and demographic variables needed for the time-stability analysis.

### Step 11 | Save the Cleaned Dataset

Finally, we save the cleaned multi-year dataset for use in the regression and interpretation stages of RQ3.

In [43]:
output_path = os.path.join(downloads_path, "hmda_2022_2025_cleaned_for_rq3.csv")
rq3_df.to_csv(output_path, index=False)

print("Saved cleaned file to:")
print(output_path)

Saved cleaned file to:
/Users/brian/Downloads/hmda_2022_2025_cleaned_for_rq3.csv


### Step 12 | Final Note

The resulting file will be used in the next stage of RQ3, where we compare mortgage approval determinants and demographic disparities across time periods using regression models.